In [1]:
%load_ext autoreload
%autoreload 2
# Always Restart Kernel after modifying backend file
from backend_aipw import *
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from econml.grf import RegressionForest
from sklearn.model_selection import KFold, GridSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from scipy.special import logit, expit
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA

In [2]:
def generate_dgp3_params(rng, n, dim_factor, spars_factor=None, fixed_s=None): # Takes as input the sample size, dimensionality factor, and sparsity factor
    # Hyperparameters controlling the strength of confounding and noise
    p_band = [0.2, 0.8]
    beta_strength = 2.5 # Before 3.0
    rho_strength = 0.8 # Before 1.2
    beta_noise = 0.5
    rho_noise = 0.3
    
    # Set Dimensionality
    n_groups = int(dim_factor * n)  # Dimensionality: grows with n
    
    # Set Sparsity
    if fixed_s is not None:
        nunique = fixed_s # Fixed Sparsity -> independent of n
    else:
        nunique = int(spars_factor * n_groups) # Relative Sparsity -> varies with n
    
    s_unique = rng.normal(0, 1, nunique) # Direct Supervisor Translation of latent factor
    s = np.resize(s_unique, n_groups) # Direct Supervisor Translation of latent factor
    #s += 0.01 * rng.normal(0, 1, size=n_groups) # Add small noise to break perfect repetition
    
    # INCORRECT APPROACH: causes SVD convergence failure [Latent sparsity]
    #s = np.zeros(n_groups)
    #active_idx = rng.choice(n_groups, size=nunique, replace=False) # Controlling which groups are assigned signal. Rest of Groups have zero signal
    #s[active_idx] = rng.normal(0, 1, size=nunique)

    # Baseline outcomes (beta_g) are driven by latent 's'
    beta_g = beta_strength * s + beta_noise * rng.normal(0, 1, size=n_groups)
    # Propensity latent score (rho_latent) is ALSO driven by 's' -> Strong Confounding!
    rho_latent = rho_strength * s + rho_noise * rng.normal(0, 1, size=n_groups)

    # Bound Propensities cleanly strictly within p_band
    a = logit(p_band[0])
    b = logit(p_band[1])
    rho_g = np.clip(rho_latent, a, b)
    p_g = expit(rho_g) # convert back to probabilities

    return n_groups, beta_g, p_g

In [3]:
# --- New Supervisor-Inspired DGP 3 ---
def dgp3_supervisor(rng, n, n_groups, beta_g, p_g):
    """
    High-dimensional, strongly confounded DGP based on supervisor's R code.
    Breaks AIPW by combining sparsity with extreme correlation between 
    outcomes and propensity scores.
    """
    # Generate the Sample
    gid = rng.integers(0, n_groups, size=n)
    
    # One-hot encoding matrix [Structural sparsity]
    X = np.zeros((n, n_groups))
    X[np.arange(n), gid] = 1.0

    # Map group parameters to individuals
    mu0 = beta_g[gid]
    mu1 = mu0 + 1.0  # Constant treatment effect tau = 1.0
    p = p_g[gid]

    # Assign treatment based on confounded propensity
    D = rng.binomial(1, p)

    # Generate Final Outcomes
    sigma_y = 1.0
    Y0 = mu0 + rng.normal(0, sigma_y, size=n)
    Y1 = mu1 + rng.normal(0, sigma_y, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [4]:
"""
Types of Machine Learers:
1) Linear Learners: OLS, Ridge, Lasso, Elastic Net, Fused Lasso
2) Tree-based Learners: Random Forest, Extra Trees, Honest Forest
3) Boosted Decision-Tree Learners: Gradient Boosting, CatBoost, XGBoost
4) Kernel-based Learners: Kernel Ridge Regression, Support Vector Regression
5) Neural Networks: 
"""

learners_regime = {
    # Worst Estimator to Best Estimator
    # 1) Linear Learners
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=1.0), # Ridge with default parameters (alpha=1.0)
    "Lasso": Lasso(alpha=1.0, max_iter=10000), # Lasso with default parameters (alpha=1.0)
    "ElasticNet": ElasticNet(max_iter=10000), # Elastic Net
    "FusedLasso": FusedLasso(alpha1=0.1, alpha2=0.1), # Fused Lasso Implementation
    #"PCR": Pipeline([
    #    ("scaler", StandardScaler()),
    #    ("pca", PCA(svd_solver="auto")),
    #    ("regressor", LinearRegression()),
    #]),
    # 2) Manifold Learners
    #"KNN": KNeighborsRegressor(n_neighbors=10, weights='distance'),
    #"KNN": Pipeline([
    #    ("scaler", StandardScaler()),
    #    ("knn", KNeighborsRegressor())
    #]),
    # 2) Tree-baseed Learners
    "RF": RandomForestRegressor(n_estimators=300, random_state=123, n_jobs=1), # Random Forest
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, bootstrap=True, n_jobs=1),
    "HonestForest": RegressionForest(n_estimators=300, min_samples_leaf=5),
    # 3) Boosted Decision-Tree Learners
    "GB": GradientBoostingRegressor(n_estimators=300, random_state=123), # Gradient Boosting
    "CatBoost": CatBoostRegressor(verbose=0, random_state=123, iterations=300, thread_count=1), # CatBoost
    "XGBoost": XGBRegressor(n_estimators=300, random_state=123, n_jobs=1), # XGBoost
    # 4) Kernel-based Leaners
    "KRR_RBF": KernelRidge(kernel='rbf', alpha=1.0, gamma=0.1), # Kernel Ridge Regression
    "SVR_RBF": SVR(kernel='rbf', C=1.0, epsilon=0.1), # # Support Vector Regression
    # 5) Neural Networks
    "NeuralNet": MLPRegressor(random_state=123, max_iter=500),
}

In [5]:
# DGP 3
# MC Parameters 
n = 100 # Sample size
dim_factor = 0.4 # Dimensionality factor: when dim_factor > 1.0 the number of groups exceeds the sample size, creating a high-dimensional setting 
#spars_factor = 0.5 # Sparsity factor: when spars_factor = 1.0, there is no sparsity
constant_fixed_sparsity = 10

print("DGP 3")
print("Sample Size", n)
print("Dimensionality", dim_factor)
print("Constant Fixed Sparsity", constant_fixed_sparsity)
n_groups, beta_g, p_g = generate_dgp3_params(np.random.default_rng(0), n=n, dim_factor=dim_factor, spars_factor=None, fixed_s = constant_fixed_sparsity)
tuned_learners_3 = tune_once_parallel(dgp3_supervisor, learners_regime, n=n, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
print(monte_carlo_parallel(dgp3_supervisor, tuned_learners_3, n=n, sims=100, n_groups=n_groups, beta_g=beta_g, p_g=p_g))

DGP 3
Sample Size 100
Dimensionality 0.4
Constant Fixed Sparsity 10
Current Monte Carlo Simulation 0
Current Monte Carlo Simulation 1
Current Monte Carlo Simulation 3
Current Monte Carlo Simulation 2
Current Monte Carlo Simulation 4
Current Monte Carlo Simulation 6
Current Monte Carlo Simulation 5
Current Monte Carlo Simulation 7


/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Current Monte Carlo Simulation 8
Current Monte Carlo Simulation 9
Current Monte Carlo Simulation 10


/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Current Monte Carlo Simulation 11
Current Monte Carlo Simulation 12
Current Monte Carlo Simulation 13
Current Monte Carlo Simulation 14


...

Current Monte Carlo Simulation 79
Current Monte Carlo Simulation 80
Current Monte Carlo Simulation 81
Current Monte Carlo Simulation 82
Current Monte Carlo Simulation 83


/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Current Monte Carlo Simulation 84
Current Monte Carlo Simulation 85
Current Monte Carlo Simulation 86
Current Monte Carlo Simulation 87
Current Monte Carlo Simulation 88
Current Monte Carlo Simulation 89
Current Monte Carlo Simulation 90
Current Monte Carlo Simulation 91
Current Monte Carlo Simulation 92
Current Monte Carlo Simulation 93
Current Monte Carlo Simulation 94
Current Monte Carlo Simulation 95
Current Monte Carlo Simulation 96
Current Monte Carlo Simulation 97
Current Monte Carlo Simulation 98
Current Monte Carlo Simulation 99


/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


                  Mean      Bias  Variance  Mean_SD_Err      RMSE  CI_Width  \
Learner                                                                       
OLS           1.323044  0.323044  0.127050     0.345704  0.479726  1.355158   
Ridge         1.395952  0.395952  0.123190     0.351726  0.527955  1.378767   
Lasso         1.384647  0.384647  0.123914     0.352686  0.520220  1.382528   
ElasticNet    1.396031  0.396031  0.123156     0.353279  0.527981  1.384855   
FusedLasso    1.330498  0.330498  0.125212     0.344778  0.482896  1.351529   
RF            1.437261  0.437261  0.123271     0.359933  0.559674  1.410939   
ExtraTrees    1.432577  0.432577  0.127791     0.366143  0.560032  1.435281   
HonestForest  1.719694  0.719694  0.144005     0.429243  0.812727  1.682634   
GB            1.393757  0.393757  0.122293     0.357536  0.525467  1.401542   
CatBoost      1.387193  0.387193  0.121788     0.356309  0.520085  1.396731   
XGBoost       1.418271  0.418271  0.123929     0.357

In [6]:
# ANALYSIS - Dimensionality (1) [Constant sample size, Constant fixed sparsity]
dimensionality_sizes = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]  # Factors of n to determine number of groups
fixed_sample_size = 200
constant_fixed_sparsity = 10 # unique groups
simulations = 300
results_n = []

for dim in dimensionality_sizes:
    print(f"Running simulation for dim_factor={dim}...")
    # Define learner regime (Tuned or using robust defaults)
    n_groups, beta_g, p_g = generate_dgp3_params(np.random.default_rng(0), n=fixed_sample_size, dim_factor=dim, spars_factor=None, fixed_s=constant_fixed_sparsity)
    tuned_learners = tune_once_parallel(dgp3_supervisor, learners_regime, n=fixed_sample_size, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results = monte_carlo_parallel(dgp3_supervisor, tuned_learners, n=fixed_sample_size, sims=simulations, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results['dim_factor'] = dim
    results_n.append(batch_results.reset_index())

df_n = pd.concat(results_n)
# Create figures 
fig_cov, ax_cov = plt.subplots(figsize=(10,6))
fig_rmse, ax_rmse = plt.subplots(figsize=(10,6))
fig_ci, ax_ci = plt.subplots(figsize=(10,6))
fig_mean, ax_mean = plt.subplots(figsize=(10,6))

for learner in df_n['Learner'].unique():
    subset = df_n[df_n['Learner'] == learner].sort_values("dim_factor")
    x = subset['dim_factor']
    ax_cov.plot(x, subset['Coverage'], marker='o', label=learner)
    ax_rmse.plot(x, subset['RMSE'], marker='o', label=learner)
    ax_ci.plot(x, subset['CI_Width'], marker='o', label=learner)
    ax_mean.plot(x, subset['Mean'], marker='o', label=learner)

# Coverage formatting
ax_cov.axhline(y=0.95, color='r', linestyle='--', label='Target Coverage')
ax_cov.set_title(f"Coverage vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant fixed sparsity = {constant_fixed_sparsity}, Number of simulations = {simulations})")
ax_cov.set_xlabel("Dimensionality Factor (p/n)")
ax_cov.set_ylabel("Coverage")
ax_cov.grid(True)
ax_cov.legend()

# RMSE formatting
ax_rmse.set_title(f"RMSE vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant fixed sparsity = {constant_fixed_sparsity}, Number of simulations = {simulations})")
ax_rmse.set_xlabel("Dimensionality Factor (p/n)")
ax_rmse.set_ylabel("RMSE")
ax_rmse.grid(True)
ax_rmse.legend()

# CI_Width formatting
ax_ci.set_title(f"CI Width vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant fixed sparsity = {constant_fixed_sparsity}, Number of simulations = {simulations})")
ax_ci.set_xlabel("Dimensionality Factor (p/n)")
ax_ci.set_ylabel("CI Width")
ax_ci.grid(True)
ax_ci.legend()

# Mean formatting
ax_mean.set_title(f"Mean Estimate vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant fixed sparsity = {constant_fixed_sparsity}, Number of simulations = {simulations})")
ax_mean.set_xlabel("Dimensionality Factor (p/n)")
ax_mean.set_ylabel("Mean Estimate")
ax_mean.grid(True)
ax_mean.legend()

plt.show()

Running simulation for dim_factor=0.1...


/home/chris2003/Uni/Thesis_Management_Technology/dml-simulation/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


KeyboardInterrupt: 

In [ ]:
# ANALYSIS - Dimensionality (2) [Constant sample size, Constant relative sparsity]
dimensionality_sizes = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]  # Factors of n to determine number of groups
fixed_sample_size = 200
constant_relative_sparsity = 0.5
simulations = 300
results_n = []

for dim in dimensionality_sizes:
    print(f"Running simulation for dim_factor={dim}...")
    # Define learner regime (Tuned or using robust defaults)
    n_groups, beta_g, p_g = generate_dgp3_params(np.random.default_rng(0), n=fixed_sample_size, dim_factor=dim, spars_factor=constant_relative_sparsity)
    tuned_learners = tune_once_parallel(dgp3_supervisor, learners_regime, n=fixed_sample_size, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results = monte_carlo_parallel(dgp3_supervisor, tuned_learners, n=fixed_sample_size, sims=simulations, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results['dim_factor'] = dim
    results_n.append(batch_results.reset_index())

df_n = pd.concat(results_n)
# Create figures 
fig_cov, ax_cov = plt.subplots(figsize=(10,6))
fig_rmse, ax_rmse = plt.subplots(figsize=(10,6))
fig_ci, ax_ci = plt.subplots(figsize=(10,6))
fig_mean, ax_mean = plt.subplots(figsize=(10,6))

for learner in df_n['Learner'].unique():
    subset = df_n[df_n['Learner'] == learner].sort_values("dim_factor")
    x = subset['dim_factor']
    ax_cov.plot(x, subset['Coverage'], marker='o', label=learner)
    ax_rmse.plot(x, subset['RMSE'], marker='o', label=learner)
    ax_ci.plot(x, subset['CI_Width'], marker='o', label=learner)
    ax_mean.plot(x, subset['Mean'], marker='o', label=learner)

# Coverage formatting
ax_cov.axhline(y=0.95, color='r', linestyle='--', label='Target Coverage')
ax_cov.set_title(f"Coverage vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant relative sparsity = {constant_relative_sparsity}, Number of simulations = {simulations})")
ax_cov.set_xlabel("Dimensionality Factor (p/n)")
ax_cov.set_ylabel("Coverage")
ax_cov.grid(True)
ax_cov.legend()

# RMSE formatting
ax_rmse.set_title(f"RMSE vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant relative sparsity = {constant_relative_sparsity}, Number of simulations = {simulations})")
ax_rmse.set_xlabel("Dimensionality Factor (p/n)")
ax_rmse.set_ylabel("RMSE")
ax_rmse.grid(True)
ax_rmse.legend()

# CI_Width formatting
ax_ci.set_title(f"CI Width vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant relative sparsity = {constant_relative_sparsity}, Number of simulations = {simulations})")
ax_ci.set_xlabel("Dimensionality Factor (p/n)")
ax_ci.set_ylabel("CI Width")
ax_ci.grid(True)
ax_ci.legend()

# Mean formatting
ax_mean.set_title(f"Mean Estimate vs Dimensionality (Constant sample size = {fixed_sample_size}, Constant relative sparsity = {constant_relative_sparsity}, Number of simulations = {simulations})")
ax_mean.set_xlabel("Dimensionality Factor (p/n)")
ax_mean.set_ylabel("Mean Estimate")
ax_mean.grid(True)
ax_mean.legend()

plt.show()

In [ ]:
# ANALYSIS - Sparsity
sparsity_sizes = [0.2, 0.4, 0.6, 0.8, 1.0] # Relative Sparsity factors to determine number of unique signals
fixed_sample_size = 200
fixed_dimensionality_size = 1.0 # 0.8 was too high
simulations = 300
results_n = []

for spars in sparsity_sizes:
    print(f"Running simulation for spars_factor={spars}...")
    # Define learner regime (Tuned or using robust defaults)
    n_groups, beta_g, p_g = generate_dgp3_params(np.random.default_rng(0), n=fixed_sample_size, dim_factor=fixed_dimensionality_size, spars_factor=spars) 
    tuned_learners = tune_once_parallel(dgp3_supervisor, learners_regime, n=fixed_sample_size, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results = monte_carlo_parallel(dgp3_supervisor, tuned_learners, n=fixed_sample_size, sims=simulations, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results['spars_factor'] = spars
    results_n.append(batch_results.reset_index())

df_n = pd.concat(results_n)
# Create figures
fig_cov, ax_cov = plt.subplots(figsize=(10,6))
fig_rmse, ax_rmse = plt.subplots(figsize=(10,6))
fig_ci, ax_ci = plt.subplots(figsize=(10,6))
fig_mean, ax_mean = plt.subplots(figsize=(10,6))

for learner in df_n['Learner'].unique():
    subset = df_n[df_n['Learner'] == learner].sort_values("spars_factor")
    x = subset['spars_factor']
    ax_cov.plot(x, subset['Coverage'], marker='o', label=learner)
    ax_rmse.plot(x, subset['RMSE'], marker='o', label=learner)
    ax_ci.plot(x, subset['CI_Width'], marker='o', label=learner)
    ax_mean.plot(x, subset['Mean'], marker='o', label=learner)

# Coverage formatting
ax_cov.axhline(y=0.95, color='r', linestyle='--', label='Target Coverage')
ax_cov.set_title(f"Coverage vs Sparsity (Constant sample size = {fixed_sample_size}, Constant dimensionality = {fixed_dimensionality_size}, Number of simulations = {simulations})")
ax_cov.set_xlabel("Sparsity Factor (s/p)")
ax_cov.set_ylabel("Coverage")
ax_cov.grid(True)
ax_cov.legend()

# RMSE formatting
ax_rmse.set_title(f"RMSE vs Sparsity (Constant sample size = {fixed_sample_size}, Constant dimensionality = {fixed_dimensionality_size}, Number of simulations = {simulations})")
ax_rmse.set_xlabel("Sparsity Factor (s/p)")
ax_rmse.set_ylabel("RMSE")
ax_rmse.grid(True)
ax_rmse.legend()

# CI_Width formatting
ax_ci.set_title(f"CI Width vs Sparsity (Constant sample size = {fixed_sample_size}, Constant dimensionality = {fixed_dimensionality_size}, Number of simulations = {simulations})")
ax_ci.set_xlabel("Sparsity Factor (s/p)")
ax_ci.set_ylabel("CI Width")
ax_ci.grid(True)
ax_ci.legend()

# Mean formatting
ax_mean.set_title(f"Mean Estimate vs Sparsity (Constant sample size = {fixed_sample_size}, Constant dimensionality = {fixed_dimensionality_size}, Number of simulations = {simulations})")
ax_mean.set_xlabel("Sparsity Factor (s/p)")
ax_mean.set_ylabel("Mean Estimate")
ax_mean.grid(True)
ax_mean.legend()

plt.show()

In [ ]:
# ANALYSIS - Sample Size
sample_sizes = [50, 100, 200, 400, 800] # Sample sizes to test
fixed_dimensionality_size = 1.0 # 0.8 was too high
fixed_sparsity_size = 0.5
simulations = 300
results_n = []

for n in sample_sizes:
    print(f"Running simulation for sample_size={n}...")
    # Define learner regime (Tuned or using robust defaults)
    n_groups, beta_g, p_g = generate_dgp3_params(np.random.default_rng(0), n=n, dim_factor=fixed_dimensionality_size, spars_factor=fixed_sparsity_size) 
    tuned_learners = tune_once_parallel(dgp3_supervisor, learners_regime, n=n, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results = monte_carlo_parallel(dgp3_supervisor, tuned_learners, n=n, sims=simulations, n_groups=n_groups, beta_g=beta_g, p_g=p_g)
    batch_results['sample_size'] = n
    results_n.append(batch_results.reset_index())

df_n = pd.concat(results_n)
# Create figures
fig_cov, ax_cov = plt.subplots(figsize=(10,6))
fig_rmse, ax_rmse = plt.subplots(figsize=(10,6))
fig_ci, ax_ci = plt.subplots(figsize=(10,6))
fig_mean, ax_mean = plt.subplots(figsize=(10,6))

for learner in df_n['Learner'].unique():
    subset = df_n[df_n['Learner'] == learner].sort_values("sample_size")
    x = subset['sample_size']
    ax_cov.plot(x, subset['Coverage'], marker='o', label=learner)
    ax_rmse.plot(x, subset['RMSE'], marker='o', label=learner)
    ax_ci.plot(x, subset['CI_Width'], marker='o', label=learner)
    ax_mean.plot(x, subset['Mean'], marker='o', label=learner)

# Coverage formatting
ax_cov.axhline(y=0.95, color='r', linestyle='--', label='Target Coverage')
ax_cov.set_title(f"Coverage vs Sample Size (Constant dimensionality = {fixed_dimensionality_size}, Constant sparsity = {fixed_sparsity_size}, Number of simulations = {simulations})")
ax_cov.set_xlabel("Sample Size")
ax_cov.set_ylabel("Coverage")
ax_cov.grid(True)
ax_cov.legend()

# RMSE formatting
ax_rmse.set_title(f"RMSE vs Sample Size (Constant dimensionality = {fixed_dimensionality_size}, Constant sparsity = {fixed_sparsity_size}, Number of simulations = {simulations})")
ax_rmse.set_xlabel("Sample Size")
ax_rmse.set_ylabel("RMSE")
ax_rmse.grid(True)
ax_rmse.legend()

# CI_Width formatting
ax_ci.set_title(f"CI Width vs Sample Size (Constant dimensionality = {fixed_dimensionality_size}, Constant sparsity = {fixed_sparsity_size}, Number of simulations = {simulations})")
ax_ci.set_xlabel("Sample Size")
ax_ci.set_ylabel("CI Width")
ax_ci.grid(True)
ax_ci.legend()

# Mean formatting
ax_mean.set_title(f"Mean Estimate vs Sample Size (Constant dimensionality = {fixed_dimensionality_size}, Constant sparsity = {fixed_sparsity_size}, Number of simulations = {simulations})")
ax_mean.set_xlabel("Sample Size")
ax_mean.set_ylabel("Mean Estimate")
ax_mean.grid(True)
ax_mean.legend()

plt.show()
